In [21]:
# 07_adaptive_k_rerank.ipynb
# Adaptive-K candidate budgeting for two-stage MLP cosine + exact WJ rerank.
#
# Idea:
#   Easy query  -> rerank fewer candidates
#   Hard query  -> rerank more candidates
#
# Start on 10k. If it gives a good recall/QPS tradeoff, reuse the same notebook on full.


In [22]:
import os
import pickle
import random
import time

import nmslib
import numpy as np
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

THREADS = 32
QUERY_START_10K = 8000
QUERY_START_FULL = 187019

class QuadtreeCompressorV1(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x):
        return self.net(x)

class QuadtreeCompressorV1Fixed(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x):
        x = torch.log1p(x * 1e6)
        return self.net(x)

def get_mem_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

def generate_embeddings(model, data, device, batch_size=512):
    model.eval()
    chunks = []
    with torch.no_grad():
        for start in tqdm(range(0, len(data), batch_size), desc="Embedding"):
            batch = torch.tensor(data[start:start + batch_size], dtype=torch.float32, device=device)
            chunks.append(F.normalize(model(batch), dim=1).cpu().numpy())
    return np.vstack(chunks)

def build_cosine_index(corpus_embs, ef_search=200):
    m0 = get_mem_mb()
    idx = nmslib.init(method="hnsw", space="cosinesimil")
    for i in tqdm(range(len(corpus_embs)), desc="Adding", mininterval=2.0):
        idx.addDataPoint(i, corpus_embs[i])
    t0 = time.time()
    idx.createIndex({"M": 20, "efConstruction": 200, "post": 1}, print_progress=True)
    build_s = time.time() - t0
    idx_mb = get_mem_mb() - m0
    idx.setQueryTimeParams({"efSearch": ef_search})
    return idx, build_s, idx_mb

def recall_at_k_for_ids(gt_lookup, ids, qid, k):
    gt = set(gt_lookup.get(qid, [])[:k])
    if not gt:
        return np.nan
    return len(gt & set(ids[:k])) / len(gt)

def aggregate_recall(gt_lookup, reranked, query_start, k):
    vals = []
    for i, ids in enumerate(reranked):
        r = recall_at_k_for_ids(gt_lookup, ids, query_start + i, k)
        if r == r:
            vals.append(r)
    return float(np.mean(vals)) if vals else 0.0

def cosine_gap_features(nbrs, probe_k=100):
    feats = []
    for ids, dists in nbrs:
        d = np.asarray(dists, dtype=np.float32)
        if len(d) < probe_k:
            padded = np.pad(d, (0, probe_k - len(d)), constant_values=d[-1] if len(d) else 1.0)
            d = padded
        # nmslib cosinesimil returns distance-like values sorted ascending.
        d1, d10, d50, d100 = d[0], d[9], d[49], d[probe_k - 1]
        feats.append({
            "d1": float(d1),
            "gap_10": float(d10 - d1),
            "gap_50": float(d50 - d1),
            "gap_100": float(d100 - d1),
            "std_100": float(np.std(d[:probe_k])),
        })
    return feats


In [ ]:
# Configuration
dataset_name = "10k"          # "10k" first, then "full"
model_variant = "harddist"    # "base" or "harddist"
device = torch.device("cuda:0")
seed = 123

candidate_ks = [100, 200, 500, 1000]
probe_k = 100
max_k = max(candidate_ks)
calibration_frac = 0.5
# Aggregate calibration target on the calibration split.
target_r100 = 0.998

# For full, start with batch 4 or 8 depending on GPU memory.
rerank_batch_size = 16
out_path = "/tmp/results_adaptive_k.pkl"

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


In [24]:
# Load data and selected model
if dataset_name == "10k":
    qt = np.load("/tmp/qt_10k.npy")
    with open("/tmp/gt_lookup_10k.pkl", "rb") as f:
        gt = pickle.load(f)
    query_start = QUERY_START_10K
    model_cls = QuadtreeCompressorV1
    base_ckpt = "/tmp/best_compressor_v1_clean.pt"
    hard_ckpt = "/tmp/best_compressor_hardneg_wjdistill_10k.pt"
elif dataset_name == "full":
    qt = np.load("/tmp/qtree_vectors_full.npy")
    with open("/tmp/gt_lookup_full.pkl", "rb") as f:
        gt = pickle.load(f)
    query_start = QUERY_START_FULL
    model_cls = QuadtreeCompressorV1Fixed
    base_ckpt = "/tmp/best_compressor_full_fixed.pt"
    hard_ckpt = "/tmp/best_compressor_hardneg_wjdistill_full.pt"
else:
    raise ValueError(dataset_name)

ckpt = hard_ckpt if model_variant == "harddist" and os.path.exists(hard_ckpt) else base_ckpt
model = model_cls(qt.shape[1], out_dim=512).to(device)
model.load_state_dict(torch.load(ckpt, weights_only=True, map_location=device))
model.eval()

corpus_qt = qt[:query_start]
query_qt = qt[query_start:]
corpus_sums = corpus_qt.sum(axis=1)
print(f"dataset={dataset_name} | model_variant={model_variant} | checkpoint={ckpt}")
print(f"corpus={corpus_qt.shape} | queries={query_qt.shape}")


dataset=10k | model_variant=harddist | checkpoint=/tmp/best_compressor_hardneg_wjdistill_10k.pt
corpus=(8000, 18499) | queries=(2000, 18499)


In [25]:
# Build candidate index and mine max-K cosine candidates once for calibration/evaluation.
embs = generate_embeddings(model, qt, device)
corpus_embs = embs[:query_start]
query_embs = embs[query_start:]
idx, build_s, idx_mb = build_cosine_index(corpus_embs)

print(f"Querying max_k={max_k} for all queries...")
t0 = time.time()
nbrs_max = idx.knnQueryBatch(query_embs, k=max_k, num_threads=THREADS)
max_query_s = time.time() - t0
features = cosine_gap_features(nbrs_max, probe_k=probe_k)
print(f"max-K query time={max_query_s:.2f}s | queries={len(nbrs_max)}")


Adding: 100%|██████████| 8000/8000 [00:00<00:00, 715782.07it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

Querying max_k=1000 for all queries...
max-K query time=0.11s | queries=2000


In [26]:
# Compute exact WJ scores for max-K candidates on GPU.
def compute_wj_scores_gpu(query_qt, nbrs, corpus_qt, corpus_sums, device, batch_size=16):
    corpus_t = torch.from_numpy(corpus_qt).to(device=device, dtype=torch.float32)
    corpus_sums_t = torch.from_numpy(corpus_sums).to(device=device, dtype=torch.float32)
    all_ids, all_scores = [None] * len(nbrs), [None] * len(nbrs)

    for start in tqdm(range(0, len(nbrs), batch_size), desc="GPU WJ scoring"):
        batch = nbrs[start:start + batch_size]
        groups = {}
        for offset, (ids, _) in enumerate(batch):
            ids_arr = np.asarray(ids, dtype=np.int64)
            groups.setdefault(len(ids_arr), []).append((start + offset, ids_arr))

        for _, items in groups.items():
            ids_np = np.stack([ids for _, ids in items], axis=0)
            query_np = np.stack([query_qt[i] for i, _ in items], axis=0)
            ids_t = torch.from_numpy(ids_np).to(device=device)
            q_t = torch.from_numpy(query_np).to(device=device, dtype=torch.float32)
            c_t = corpus_t[ids_t]
            mins = torch.minimum(q_t[:, None, :], c_t).sum(dim=2)
            maxs = q_t.sum(dim=1, keepdim=True) + corpus_sums_t[ids_t] - mins
            scores = (mins / maxs.clamp_min(1e-10)).cpu().numpy()
            for row, (absolute_i, ids) in enumerate(items):
                all_ids[absolute_i] = ids
                all_scores[absolute_i] = scores[row]

    del corpus_t, corpus_sums_t
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return all_ids, all_scores

t0 = time.time()
all_ids, all_wj_scores = compute_wj_scores_gpu(query_qt, nbrs_max, corpus_qt, corpus_sums, device, rerank_batch_size)
wj_score_s = time.time() - t0
print(f"WJ scoring time={wj_score_s:.2f}s")


GPU WJ scoring: 100%|██████████| 125/125 [00:00<00:00, 143.71it/s]

WJ scoring time=0.93s


In [27]:
# Fixed-K recall from the same max-K candidate lists.
def reranked_ids_for_k(i, k):
    ids = all_ids[i][:k]
    scores = all_wj_scores[i][:k]
    order = np.argsort(-scores)
    return ids[order].tolist()

def per_query_r100_for_k(i, k):
    if k < 100:
        return np.nan
    ids = reranked_ids_for_k(i, k)
    return recall_at_k_for_ids(gt, ids, query_start + i, 100)

fixed_results = {}
for k in candidate_ks:
    reranked = [reranked_ids_for_k(i, k) for i in range(len(all_ids))]
    fixed_results[k] = {
        10: aggregate_recall(gt, reranked, query_start, 10),
        50: aggregate_recall(gt, reranked, query_start, 50),
        100: aggregate_recall(gt, reranked, query_start, 100),
        500: aggregate_recall(gt, reranked, query_start, 500) if k >= 500 else np.nan,
        "avg_k": k,
    }
    print(k, fixed_results[k])


100 {10: 0.9962733773377337, 50: 0.9724437184487459, 100: 0.8640920298105405, 500: nan, 'avg_k': 100}
200 {10: 0.9966446644664466, 50: 0.9973705747864813, 100: 0.9861486588962406, 500: nan, 'avg_k': 200}
500 {10: 0.9966446644664466, 50: 0.9985588558855886, 100: 0.9989108910891089, 500: 0.967899311485781, 'avg_k': 500}
1000 {10: 0.9966446644664466, 50: 0.9985588558855886, 100: 0.9989273927392739, 500: 0.9893851964135644, 'avg_k': 1000}


In [28]:
# Calibrate a simple adaptive-K rule from probe features.
# Higher gap_100 means the query has a clearer nearest-neighbor structure, so it can use smaller K.
n = len(all_ids)
indices = np.arange(n)
np.random.default_rng(seed).shuffle(indices)
cal_n = int(n * calibration_frac)
cal_idx = indices[:cal_n]
test_idx = indices[cal_n:]
feat = np.array([f["gap_100"] for f in features], dtype=np.float32)

# Precompute per-query R@100 for each candidate K.
r100 = {k: np.array([per_query_r100_for_k(i, k) for i in range(n)], dtype=np.float32) for k in candidate_ks}

def choose_k_for_feature(values, thresholds):
    t100, t200, t500 = thresholds
    out = np.empty(len(values), dtype=np.int32)
    out[values >= t100] = 100
    out[(values < t100) & (values >= t200)] = 200
    out[(values < t200) & (values >= t500)] = 500
    out[values < t500] = 1000
    return out

def aggregate_r100_for_choices(chosen, split_idx):
    vals = []
    for i in split_idx:
        vals.append(r100[int(chosen[i])][i])
    return float(np.nanmean(vals))

def avg_k_for_choices(chosen, split_idx):
    return float(np.mean(chosen[split_idx]))

qs = np.quantile(feat[cal_idx], np.linspace(0.05, 0.95, 19))
best = None
all_rules = []
for t100 in qs:
    for t200 in qs:
        for t500 in qs:
            if not (t100 >= t200 >= t500):
                continue
            chosen = choose_k_for_feature(feat, (t100, t200, t500))
            cal_r = aggregate_r100_for_choices(chosen, cal_idx)
            cal_avg_k = avg_k_for_choices(chosen, cal_idx)
            test_r = aggregate_r100_for_choices(chosen, test_idx)
            test_avg_k = avg_k_for_choices(chosen, test_idx)
            rec = {"thresholds": (float(t100), float(t200), float(t500)),
                   "cal_r100": cal_r, "cal_avg_k": cal_avg_k,
                   "test_r100": test_r, "test_avg_k": test_avg_k}
            all_rules.append(rec)
            if cal_r >= target_r100:
                if best is None or cal_avg_k < best["cal_avg_k"]:
                    best = rec

if best is None:
    # Fall back to best recall per average-K score.
    best = max(all_rules, key=lambda x: (x["cal_r100"] - 0.0002 * x["cal_avg_k"]))

chosen_k = choose_k_for_feature(feat, best["thresholds"])
print("Best adaptive rule:", best)
unique, counts = np.unique(chosen_k, return_counts=True)
print("Chosen K distribution:", dict(zip(unique.tolist(), counts.tolist())))


Best adaptive rule: {'thresholds': (0.20931488275527954, 0.011339938640594485, 0.0021548479795455936), 'cal_r100': 0.9900824427604675, 'cal_avg_k': 275.0, 'test_r100': 0.9906396865844727, 'test_avg_k': 285.1}
Chosen K distribution: {100: 182, 200: 1397, 500: 317, 1000: 104}


In [29]:
# Evaluate adaptive choices from cached max-K candidates.
def adaptive_reranked_ids(i):
    return reranked_ids_for_k(i, int(chosen_k[i]))

adaptive_reranked = [adaptive_reranked_ids(i) for i in range(len(all_ids))]
adaptive_results = {
    10: aggregate_recall(gt, adaptive_reranked, query_start, 10),
    50: aggregate_recall(gt, adaptive_reranked, query_start, 50),
    100: aggregate_recall(gt, adaptive_reranked, query_start, 100),
    500: aggregate_recall(gt, adaptive_reranked, query_start, 500),
    "avg_k": float(np.mean(chosen_k)),
}

print("\nFixed-K vs Adaptive-K from cached max-K candidates")
print(f"{'Method':<12} {'R@10':>7} {'R@50':>7} {'R@100':>7} {'R@500':>7} {'AvgK':>8}")
print("-" * 60)
for k, res in fixed_results.items():
    r500 = f"{res[500]:>7.4f}" if res[500] == res[500] else f"{'-':>7}"
    print(f"K={k:<9} {res[10]:>7.4f} {res[50]:>7.4f} {res[100]:>7.4f} {r500} {res['avg_k']:>8.1f}")
r500 = f"{adaptive_results[500]:>7.4f}" if adaptive_results[500] == adaptive_results[500] else f"{'-':>7}"
print(f"{'Adaptive':<12} {adaptive_results[10]:>7.4f} {adaptive_results[50]:>7.4f} {adaptive_results[100]:>7.4f} {r500} {adaptive_results['avg_k']:>8.1f}")



Fixed-K vs Adaptive-K from cached max-K candidates
Method          R@10    R@50   R@100   R@500     AvgK
------------------------------------------------------------
K=100        0.9963  0.9724  0.8641       -    100.0
K=200        0.9966  0.9974  0.9861       -    200.0
K=500        0.9966  0.9986  0.9989  0.9679    500.0
K=1000       0.9966  0.9986  0.9989  0.9894   1000.0
Adaptive      0.9966  0.9976  0.9904  0.8134    280.1


In [30]:
# Save adaptive-K calibration/evaluation results.
run_key = time.strftime(f"{dataset_name}_{model_variant}_adaptive_%Y%m%d_%H%M%S")
record = {
    "config": {
        "dataset_name": dataset_name,
        "model_variant": model_variant,
        "checkpoint": ckpt,
        "candidate_ks": candidate_ks,
        "probe_k": probe_k,
        "target_r100": target_r100,
        "rerank_batch_size": rerank_batch_size,
    },
    "build_s": build_s,
    "idx_mb": idx_mb,
    "max_query_s": max_query_s,
    "wj_score_s": wj_score_s,
    "best_rule": best,
    "chosen_k_distribution": dict(zip(unique.tolist(), counts.tolist())),
    "fixed_results": fixed_results,
    "adaptive_results": adaptive_results,
}
try:
    with open(out_path, "rb") as f:
        saved = pickle.load(f)
except FileNotFoundError:
    saved = {"runs": {}}
saved.setdefault("runs", {})[run_key] = record
with open(out_path, "wb") as f:
    pickle.dump(saved, f)
print(f"Saved run {run_key} to {out_path}")


Saved run 10k_harddist_adaptive_20260427_134710 to /tmp/results_adaptive_k.pkl
